# 第一步：智慧潍苑 App 扫码登录
使用本 notebook 生成的二维码扫码，以便 Cookie 保留在同一个 Python 会话中。二维码不是之前浏览器里的那一张。只调用登录接口，不提交预约。请勿将登录后的输出或凭证提交到 Git。

In [ ]:
import importlib
import requests
import qr_login
from IPython.display import Image, display

importlib.reload(qr_login)
login = qr_login.QRLogin(proxy="http://127.0.0.1:7890")  # 显式代理
print("正在通过代理获取二维码，请稍候……", flush=True)
try:
    qr_png = login.start()
except requests.exceptions.RequestException as exc:
    print(f"获取二维码失败：{type(exc).__name__}。当前使用代理 http://127.0.0.1:7890。")
    print("若长时间显示 [*]，请检查代理端口 7890 是否可用。")
else:
    display(Image(data=qr_png, format="png"))
    print("二维码已生成。运行等待单元格，再用 App 扫码确认。")

用智慧潍苑 App 扫描上面的二维码，然后运行下一个单元格并在手机端确认。最多等待 120 秒，可中断。

In [ ]:
result = login.wait_for_scan(seconds=120)
print(result)

登录 Cookie 保存在 `login.session`，URL 中实际出现的 token 保存在 `login.tokens`。默认只显示字段名；不要直接打印完整凭证。若状态为 `inspect_redirect` 或未出现预期 token，先检查跳转页面再继续，不猜测凭证。

## 回调报错后：只诊断，不重新扫码
若当前内核仍有 `login`，运行下面的单元格即可检查上次响应。不会发出网络请求，也不会重放授权码。

In [ ]:
import importlib
import qr_login
importlib.reload(qr_login)
print(qr_login.response_diagnostic(login.last_response))

In [ ]:
from login_diagnostics import enable_login_logging

print(enable_login_logging(login))

In [ ]:
from pathlib import Path
from login_diagnostics import scrub

path = Path("work/callback_page_redacted.html")
path.write_text(
    scrub(login.last_response.text, login.last_response),
    encoding="utf-8",
)
print(path.resolve())

In [ ]:
from web_login_step import complete_web_login

result = complete_web_login(login)
print(result)